# Import Required Libraries

In [1]:
# Import basic packages
import os
import warnings
warnings.filterwarnings("ignore")
import datetime
import re
from IPython.display import display, Markdown

# Basic ds packages
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from tqdm import tqdm

# decorator packages
from typing import List, Dict
import warnings
warnings.filterwarnings("ignore")

#sklearn packages
from sklearn.base import BaseEstimator, TransformerMixin

# Tensorflow packages
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.layers import (Dense,
                                    Layer,
                                    BatchNormalization)
from tensorflow.keras.callbacks import (EarlyStopping, 
                                        ModelCheckpoint)
from tensorflow.keras.optimizers import (Adam, 
                                         AdamW, 
                                         RMSprop)
from tensorflow.keras.losses import (SparseCategoricalCrossentropy,
                                     CategoricalCrossentropy)


# Tensorflow Text Packages
import nltk
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud, STOPWORDS
from tensorflow.keras.preprocessing.text import Tokenizer


# GenerateTrainingData

#### Step 0: Custom UDFs

In [2]:
class GenerateTrainingSamples:

    def __init__(self, context_size):
        self.context_size = context_size
        self.mid_point = int(self.context_size/2)

    def __call__(self, training_corpus:Dict):
        self.training_corpus = training_corpus

        training_points = []

        for single_context in tqdm(training_corpus.values()):
            context_training_len = len(single_context)-(self.context_size + 1)
            for iter in range(0, context_training_len):
                
                # Indexing values based on context size
                left_end_split = iter+self.mid_point
                right_start_split = left_end_split + 1
                right_end_split = right_start_split + self.mid_point
                
                # Training Points
                training_points.append([single_context[iter:left_end_split] + \
                                        single_context[right_start_split:right_end_split],\
                                        single_context[right_end_split]])
                
        return training_points

#### 1. Reading Corpus

In [3]:
with open("../artifacts/corpus/text_corpus.pkl", "rb") as f:
    lemma_corpus = pickle.load(f)
    print("Done!! Reading the Lemma Corpus")

Done!! Reading the Lemma Corpus


In [4]:
lemma_corpus_subset = {key: lemma_corpus[key] for key in range(0, 100)}

#### Step 2 : Generate Training Samples

In [5]:
generator = GenerateTrainingSamples(context_size=4)
training_samples = generator(lemma_corpus_subset)

with open("../artifacts/trainingData/training_samples.pkl","wb") as f:
    pickle.dump(training_samples, f)
    print("Done!! writing training samples")

100%|██████████| 100/100 [00:00<00:00, 12528.54it/s]

Done!! writing training samples


In [6]:
with open("../artifacts/trainingData/training_samples.pkl","rb") as f:
    training_samples = pickle.load(f)

#### Step 3: Generate Vocabulary

In [7]:
vocabulary_corpus = []
for numpy_val  in tqdm(lemma_corpus_subset.values()):
    vocabulary_corpus.extend(numpy_val)

print(f"Get Unique Vocabulary Size : {len(set(vocabulary_corpus))}")
unique_vocabulary_corpus = set(vocabulary_corpus)

100%|██████████| 100/100 [00:00<00:00, 768187.55it/s]

Get Unique Vocabulary Size : 2006


#### Step 4: OneHotEncoding

In [8]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(np.array(list(unique_vocabulary_corpus)).reshape(-1,1))

LabelEncoder()

In [9]:
# Save the fitted model
with open("../artifacts/preprocessing/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)
    print("Done!! Writting the LabelEncoder")

Done!! Writting the LabelEncoder


In [10]:
# Load the fitted model
with open("../artifacts/preprocessing/label_encoder.pkl", "rb") as f:
    le = pickle.load(f)
    print("Done!! Reading the LabelEncoder")

Done!! Reading the LabelEncoder


#### Step 5: Training Sample with Label Encoded Value

In [82]:
def num_repre(training_samples : List[List[str]], le) -> np.array:
    training_X_samples = np.array([])
    training_y_samples = np.array([])
    
    for i in tqdm(range(0, len(training_samples))):
        
        # Input Features
        input_se = le.transform(np.array(training_samples[i][0]).reshape(-1,1)) 
        
        # Output Features
        output_se = le.transform(np.array(training_samples[i][1]).reshape(-1,1)) 

        # Add features to training ready samples list
        if training_X_samples.shape[0] == 0:
            training_X_samples = input_se
        else:
            training_X_samples = np.vstack((training_X_samples, input_se))


        if training_y_samples.shape[0] == 0:
            training_y_samples = output_se
        else:
            training_y_samples = np.vstack((training_y_samples, output_se))


    return training_X_samples, training_y_samples

In [83]:
training_ready_samples = num_repre(training_samples= training_samples, 
                                   le= le)

100%|██████████| 7450/7450 [00:18<00:00, 400.27it/s]


In [87]:
training_ready_samples[0]

array([[1756, 1632, 1894,  189],
       [1632, 1801,  189,  931],
       [1801, 1894,  931, 1245],
       ...,
       [1736, 1801, 1431, 1217],
       [1801,  139, 1217, 1176],
       [ 139, 1431, 1176,  929]])

In [88]:
training_ready_samples[1]

array([[ 931],
       [1245],
       [1736],
       ...,
       [1176],
       [ 929],
       [ 928]])

#### Step 6: Saving the Numerical Representation of Training Samples

In [86]:
with open("../artifacts/trainingData/training_ready_samples.pkl", "wb") as f:
    pickle.dump(training_ready_samples, f)
    print("Done!! Loading Training Ready Samples")

Done!! Loading Training Ready Samples
